In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder.appName("BrawlAnalysis").getOrCreate()
sc = spark.sparkContext

In [4]:
#Importing CSVs
legends_set = spark.read.csv(r"C:\Users\0106o\OneDrive\Pictures\Documents\Datasets\Brawlhalla\brawlmance-legends.csv", header = True, inferSchema = True)
weapons_set = spark.read.csv(r"C:\Users\0106o\OneDrive\Pictures\Documents\Datasets\Brawlhalla\brawlmance-weapons.csv", header = True, inferSchema = True)

In [5]:
#Legends Data
legends_set.show()


+---------+----------------+--------+---------+-------+-----+-----------+----------+------+--------+-------+--------+-----------+-----------+-------------------+-------------------+---------------------+---------------------+---------+-----------------+-------------------+-------------------+
|legend_id|        bio_name|strength|dexterity|defense|speed| weapon_one|weapon_two| games|playrate|winrate|suicides|damagetaken|damagedealt|damagedealt_unarmed|damagedealt_gadgets|damagedealt_weaponone|damagedealt_weapontwo|matchtime|matchtime_unarmed|matchtime_weaponone|matchtime_weapontwo|
+---------+----------------+--------+---------+-------+-----+-----------+----------+------+--------+-------+--------+-----------+-----------+-------------------+-------------------+---------------------+---------------------+---------+-----------------+-------------------+-------------------+
|       35|          Mordex|       6|        4|      5|    7|     Scythe|     Fists|295740|    4.61|  48.19|    0.17| 

In [6]:
# Weapons dataset
weapons_set.show()

+-----------+-------+--------+-------+------------+----------+--------+
|  weapon_id|legends|playrate|winrate|damage_dealt|match_time|timeheld|
+-----------+-------+--------+-------+------------+----------+--------+
|      Fists|     11|    2.17|  49.29|      182.57|     170.6|   62.73|
|     Pistol|     12|    1.79|  49.79|      199.45|    170.66|   67.59|
|     Scythe|     10|    1.77|  49.05|      210.42|    171.85|   72.48|
|      Spear|     13|    1.66|  50.02|      211.15|    170.21|   64.19|
|     Hammer|     11|     1.5|  50.57|      205.46|    165.47|   64.29|
|     Cannon|      7|    1.47|  51.27|      242.67|    168.34|   72.21|
|      Sword|     13|    1.41|  49.91|       165.7|    169.57|   56.46|
|        Bow|     10|     1.4|  49.66|      200.08|    168.64|    62.4|
|        Axe|     11|    1.33|  50.36|      181.43|    166.61|   55.74|
|      Boots|      5|    1.26|  49.48|      189.28|    168.47|    68.5|
|RocketLance|      9|    1.02|  50.87|       208.1|    166.92|  

In [7]:
#Q1A: What are the top 5 legends by play rate?
playrate = legends_set.select(["bio_name", "playrate"])
playrate.orderBy(F.desc("playrate")).limit(5).show()

+--------+--------+
|bio_name|playrate|
+--------+--------+
|  Mordex|    4.61|
|     Nix|     3.7|
|Wu Shang|     3.7|
|   Diana|    3.21|
|   Teros|    2.96|
+--------+--------+



In [8]:
#Q1B: What are the top 5 legends by win rate?
winrate = legends_set.select(["bio_name", "winrate"])
winrate.orderBy(F.desc("winrate")).limit(5).show()

+----------------+-------+
|        bio_name|winrate|
+----------------+-------+
|Qinghua & Baobao|  52.26|
|          Magyar|  52.24|
|      Sir Roland|  52.17|
|            Thor|  52.12|
|           Imugi|  51.96|
+----------------+-------+



In [9]:
#Q1C: Win rate and play rate product
legends_set = legends_set.withColumn("win_play_product", F.round((legends_set["winrate"])*(legends_set["playrate"]),2))
win_play_product = legends_set.select(["bio_name", "win_play_product"])

In [10]:
win_play_product.orderBy(win_play_product["win_play_product"].desc()).show()

+----------------+----------------+
|        bio_name|win_play_product|
+----------------+----------------+
|          Mordex|          222.16|
|             Nix|          183.56|
|        Wu Shang|           180.6|
|           Diana|          159.41|
|           Teros|          148.83|
|             Ada|          138.34|
|          Yumiko|          127.13|
|           Tezca|          126.68|
|           Sidra|          121.72|
|             Val|          109.37|
|           Orion|          103.14|
|           Brynn|           102.9|
|          Bödvar|          102.23|
|         Hattori|          100.57|
|          Zariel|          100.24|
|            Onyx|           98.32|
|Qinghua & Baobao|           94.07|
|         Cassidy|           93.65|
|          Mirage|           93.46|
|           Cross|           85.92|
+----------------+----------------+
only showing top 20 rows


In [11]:
#Q2A: Top 5 legends for most damage dealt
damage = legends_set.select(["bio_name","damagedealt"])
damage.orderBy(F.desc("damagedealt")).limit(5).show()

+----------------+-----------+
|        bio_name|damagedealt|
+----------------+-----------+
|          Yumiko|     485.32|
|         Arcadia|     484.82|
|       Queen Nai|     482.49|
|           Imugi|     482.06|
|Qinghua & Baobao|     478.92|
+----------------+-----------+



In [12]:
#Q2B: Top 5 legends for least damage taken
damage = legends_set.select(["bio_name","damagetaken"])
damage.orderBy(F.asc("damagetaken")).limit(5).show()

+---------+-----------+
| bio_name|damagetaken|
+---------+-----------+
|     Thea|     396.29|
|  Rupture|     398.29|
|     Loki|     398.39|
|King Zuva|     402.67|
|    Munin|     403.58|
+---------+-----------+



In [13]:
#Q2C: Damage Taken To Damage Dealt Ratio
legends_set = legends_set.withColumn("damage_ratio", (legends_set["damagedealt"])/(legends_set["damagetaken"]))
damage_ratio = legends_set.select(["bio_name", "damage_ratio"]).withColumn("damage_ratio", F.round(F.col("damage_ratio"),3))


In [14]:
damage_ratio.orderBy(damage_ratio["damage_ratio"].desc()).show()

+----------------+------------+
|        bio_name|damage_ratio|
+----------------+------------+
|          Yumiko|       1.182|
|           Imugi|       1.177|
|       Queen Nai|       1.161|
|         Arcadia|       1.158|
|Qinghua & Baobao|       1.152|
|           Seven|       1.144|
|          Magyar|        1.13|
|           Aurus|       1.129|
|            Xull|       1.126|
|            Dusk|       1.111|
|          Jaeyun|       1.108|
|           Azoth|       1.108|
|       Lady Vera|       1.106|
|         Rupture|       1.106|
|         Scarlet|       1.102|
|           Petra|       1.101|
|          Ransom|       1.101|
|           Orion|       1.091|
|            Onyx|        1.09|
|           Priya|        1.09|
+----------------+------------+
only showing top 20 rows


In [15]:
#Q3 Among the top 15 most played legends, which weapons are the most common
first_weapon = legends_set.orderBy(F.desc("playrate")).limit(15).select("weapon_one")
first_weapon = first_weapon.withColumnRenamed("weapon_one", "weapon").groupBy("weapon").count()



In [16]:
second_weapon = legends_set.orderBy(F.desc("playrate")).limit(15).select("weapon_two")
second_weapon = second_weapon.withColumnRenamed("weapon_two", "weapon").groupBy("weapon").count()

weapon_counts = first_weapon.unionByName(second_weapon)


In [17]:
weapon_counts.groupBy("weapon").agg(F.sum("count").alias("count")).orderBy(F.desc("count")).show()

+-----------+-----+
|     weapon|count|
+-----------+-----+
|      Fists|    5|
|      Spear|    5|
|      Sword|    4|
|        Bow|    3|
|     Pistol|    3|
|     Hammer|    3|
|     Scythe|    2|
|        Axe|    2|
|      Boots|    1|
|     Cannon|    1|
|RocketLance|    1|
+-----------+-----+



In [18]:
#Q5A What is the average stat line of a top 15 legend in playrate
stat_cols = ["strength","dexterity","defense","speed"]
legends_playrate_stats = legends_set.orderBy(F.desc("playrate")).limit(15).select("bio_name", *stat_cols)
legends_playrate_stats.show()



+--------+--------+---------+-------+-----+
|bio_name|strength|dexterity|defense|speed|
+--------+--------+---------+-------+-----+
|  Mordex|       6|        4|      5|    7|
|     Nix|       4|        5|      7|    6|
|Wu Shang|       5|        7|      5|    5|
|   Diana|       5|        6|      5|    6|
|   Teros|       8|        3|      6|    5|
|     Ada|       6|        7|      3|    6|
|   Tezca|       7|        5|      5|    5|
|  Yumiko|       4|        7|      4|    7|
|   Sidra|       6|        4|      6|    6|
|     Val|       4|        5|      6|    7|
|   Brynn|       5|        5|      5|    7|
| Hattori|       4|        6|      4|    8|
|  Zariel|       7|        4|      7|    4|
|  Bödvar|       6|        6|      5|    5|
|   Orion|       4|        6|      6|    6|
+--------+--------+---------+-------+-----+



In [19]:
summed_stats = legends_playrate_stats.drop("bio_name").agg(*[(F.sum(c)).alias(f"{c}") for c in stat_cols])
average_stats = summed_stats.select(*[F.round((F.col(c)/legends_playrate_stats.count()),2).alias(f"average {c}") for c in summed_stats.columns])
average_stats.show()

+----------------+-----------------+---------------+-------------+
|average strength|average dexterity|average defense|average speed|
+----------------+-----------------+---------------+-------------+
|             5.4|             5.33|           5.27|          6.0|
+----------------+-----------------+---------------+-------------+



In [20]:
#Q4B In winrate?
stat_cols = ["strength","dexterity","defense","speed"]
legends_winrate_stats = legends_set.orderBy(F.desc("winrate")).limit(15).select("bio_name", *stat_cols)

summed_stats = legends_winrate_stats.drop("bio_name").agg(*[(F.sum(c)).alias(f"{c}") for c in stat_cols])
average_stats = summed_stats.select(*[F.round((F.col(c)/legends_winrate_stats.count()),2).alias(f"average {c}") for c in summed_stats.columns])
average_stats.show()


+----------------+-----------------+---------------+-------------+
|average strength|average dexterity|average defense|average speed|
+----------------+-----------------+---------------+-------------+
|            5.93|             4.47|           6.87|         4.73|
+----------------+-----------------+---------------+-------------+



In [21]:
#Q5 Top 5 weapons by winrate and playrate
weap_cols = ["weapon_id", "playrate", "winrate", "damage_dealt", "match_time"]
weapon_playrate = weapons_set.select(weap_cols[0],weap_cols[1])
weapon_winrate = weapons_set.select(weap_cols[0],weap_cols[2])

weapon_playrate.orderBy(F.desc(weap_cols[1])).limit(5).show()

+---------+--------+
|weapon_id|playrate|
+---------+--------+
|    Fists|    2.17|
|   Pistol|    1.79|
|   Scythe|    1.77|
|    Spear|    1.66|
|   Hammer|     1.5|
+---------+--------+



In [22]:
weapon_winrate.orderBy(F.desc(weap_cols[2])).limit(5).show()

+-----------+-------+
|  weapon_id|winrate|
+-----------+-------+
|     Cannon|  51.27|
| Greatsword|  51.15|
|    Chakram|  50.91|
|RocketLance|  50.87|
|     Hammer|  50.57|
+-----------+-------+



In [23]:
#Q6A Which weapons are tied to the longest match times? 
weapon_time = weapons_set.select(weap_cols[0], weap_cols[4])
weapon_damage = weapons_set.select(weap_cols[0], weap_cols[3])
weapon_time.orderBy(F.desc(weap_cols[4])).show()

+-----------+----------+
|  weapon_id|match_time|
+-----------+----------+
|     Scythe|    171.85|
|    Chakram|    171.42|
|     Pistol|    170.66|
|      Fists|     170.6|
|        Orb|    170.55|
|      Spear|    170.21|
|      Sword|    169.57|
|        Bow|    168.64|
|      Boots|    168.47|
|     Cannon|    168.34|
|      Katar|    168.11|
| Greatsword|    167.57|
|RocketLance|    166.92|
|        Axe|    166.61|
|     Hammer|    165.47|
+-----------+----------+



In [24]:
#Q6B: The most damage?
weapon_damage.orderBy(F.desc(weap_cols[3])).show()

+-----------+------------+
|  weapon_id|damage_dealt|
+-----------+------------+
| Greatsword|      261.97|
|    Chakram|      244.63|
|     Cannon|      242.67|
|      Spear|      211.15|
|     Scythe|      210.42|
|RocketLance|       208.1|
|     Hammer|      205.46|
|        Bow|      200.08|
|     Pistol|      199.45|
|      Boots|      189.28|
|        Orb|      187.19|
|      Katar|      185.13|
|      Fists|      182.57|
|        Axe|      181.43|
|      Sword|       165.7|
+-----------+------------+



In [25]:
#Q7 Which legends in the top 15 played are overperforming relative to their weapon winrates?
weapon_slot1 = weapons_set.select(
    F.col(weap_cols[0]).alias("weapon_one"),
    F.col(weap_cols[2]).alias("weapon_one_winrate")
)
weapon_slot2 = weapons_set.select(
    F.col(weap_cols[0]).alias("weapon_two"),
    F.col(weap_cols[2]).alias("weapon_two_winrate")
)


winrate_expanded = legends_set.select(["bio_name", "playrate", "winrate", "weapon_one", "weapon_two"])

weapon_winrate = winrate_expanded.join(weapon_slot1, on = "weapon_one", how = "left").join(weapon_slot2, on = "weapon_two", how = "left")

weapon_winrate = weapon_winrate.withColumnsRenamed({"winrate":"legend_winrate", "playrate":"legend_playrate"})

weapon_winrate = weapon_winrate.withColumns(
    {"average_weapon_winrate": F.round((F.col("weapon_one_winrate") + F.col("weapon_two_winrate")) / 2, 2),
    "winrate_diff" :  F.round(F.col("legend_winrate") - F.col("average_weapon_winrate"), 2)}
    )


filtered_weapon_winrate = weapon_winrate.orderBy(F.desc("legend_playrate")).limit(10)
filtered_weapon_winrate.select(["bio_name", "legend_winrate", "average_weapon_winrate", "winrate_diff"]).orderBy(F.desc("winrate_diff")).show()

+--------+--------------+----------------------+------------+
|bio_name|legend_winrate|average_weapon_winrate|winrate_diff|
+--------+--------------+----------------------+------------+
|   Sidra|         51.36|                 50.59|        0.77|
|     Ada|         50.49|                 49.91|        0.58|
|     Nix|         49.61|                 49.42|        0.19|
|  Yumiko|         50.25|                 50.11|        0.14|
|   Diana|         49.66|                 49.72|       -0.06|
|     Val|         49.49|                  49.6|       -0.11|
|   Teros|         50.28|                 50.47|       -0.19|
|   Tezca|         48.91|                 49.39|       -0.48|
|Wu Shang|         48.81|                 49.66|       -0.85|
|  Mordex|         48.19|                 49.17|       -0.98|
+--------+--------------+----------------------+------------+

